# Konu 8 — Bir haftalık gönderi seti

**Bugünün işi**, bir sosyal medya tasarımcısının her hafta yaptığı iş. Müşterin Sayfa Kahve (üniversitenin yanındaki
sessiz çalışma kafesi) haftalık içerik takvimini gönderdi: beş gün, her gün için bir başlık ve bir sahne.
Pazartesi'ye kadar senden istenenler:

- her gün için bir **gönderi** (4:5, 1080 × 1350) ve aynı görselden bir **hikâye** (9:16, 1080 × 1920): 10 dosya
- hepsini müşterinin tek bakışta onaylayacağı bir **onay sayfası**
- müşteri "şunu değiştir" dediğinde neyi değiştireceğini bilmen için bir **üretim günlüğü**

Bir arayüzde bu, aynı işi on kez elle yapmak demek: üret, indir, şablona koy, başlığı yaz, ölçüye uyarla, kaydet.
Bugün bunu bir tablo ve bir döngüyle yapıyoruz. Ders bitince `ciktilar/` klasöründe müşteriye gidebilecek bir set
olacak. Aynı kodu ödevde kendi müşterin için kullanacaksın; o set portfolyona girer.

| İş | Kim yapıyor |
|---|---|
| Her günün görseli | yapay zekâ (5 istek) |
| Takvimi okumak, başlığı yazmak, 4:5 ve 9:16 ölçüleri, onay sayfası, günlük | Python (istek yok) |

**Çalıştırma:** hücreleri yukarıdan aşağı, sırayla çalıştır (**Shift + Enter**); çekirdek `.venv`. Isınma anahtarsız;
Adım 1'den sonrası `anahtar.txt` ister (Konu 2'deki dosya). Ana akış **7 görsel** üretir, Bonus'la 10; her görsel
ücretsiz günlük kotadan yer.

## Isınma — Takvimde ne var? (anahtarsız)

Müşterinin takvimi `veri/takvim.csv` dosyasında: her satır bir gün, sütunlar `gun`, `baslik`, `sahne`.
Konu 3'teki gibi `csv.DictReader` ile okuyoruz; her satır bir sözlük olur. `utf-8-sig`, Excel'in dosyanın başına
koyduğu görünmez işareti atlar: kendi takvimini Excel'de hazırlarsan işine yarayacak.

In [ ]:
import csv

with open("veri/takvim.csv", encoding="utf-8-sig") as dosya:
    okuyucu = csv.DictReader(dosya)
    takvim = list(okuyucu)

for satir in takvim:
    print(satir["gun"], "·", satir["baslik"])

Başlıklar Türkçe, sahneler İngilizce. Sahne görsel modeline gidecek (Konu 7: modeller İngilizceyi daha iyi anlıyor);
başlığı ise görselin üstüne Python yazacak. Sahnelerde yazı da insan yüzü de yok: Sayfa Kahve'nin kılavuzu görsellerde
yüz istemiyor, yazıyı da biz koyacağız. İstersen `veri/takvim.csv` dosyasını VS Code'da aç ve sahneleri oku.

**Tahmin et, sohbete yaz:** 5 gönderi + 5 hikâye = 10 dosya. Modele kaç istek gidecek?

In [ ]:
gun_sayisi = len(takvim)
dosya_sayisi = gun_sayisi * 2

print("Yapay zekânın üreteceği görsel:", gun_sayisi)
print("Python'un hazırlayacağı dosya:", dosya_sayisi)

Hikâye için yeni görsel üretmiyoruz: aynı görseli Python dikey ölçüye yerleştiriyor. **Üretmek kota yer, uyarlamak
yemez.** Müşteri "hikâyesini de yapıverelim" dediğinde bunun maliyeti birkaç saniyelik bilgisayar zamanı.

## Adım 1 — Pazartesi'nin gönderisi

Bütün haftaya geçmeden tek bir gönderiyi baştan sona yapıyoruz. `uretim.py` bu klasördeki yardımcı dosya; içinden
altı fonksiyon alıyoruz. Bu adımda üçünü kullanıyoruz, kalanları sırası gelince tanıtacağız:

- `ciz(yonerge, yol)`: yönergeyi görsel modeline gönderir, görseli `yol`a kaydeder; görsel gelmezse nedenini yazar ve
  `None` verir. Konu 7'deki `ciz`'den farkı: tohum yok, çünkü bu konudaki model şu an tohum kabul etmiyor
  (ne demek olduğunu Adım 2'de göreceğiz).
- `gonderi_yap(gorsel_yolu, baslik, bicim, yol)`: görseli kâğıt beyazı zemine yerleştirir, başlığı altına serifli
  fontla ve mürekkep mavisiyle yazar. Model yok, istek yok: bu Python'un işi.
- `goster(yol)`: dosyayı defterde gösterir.

İlk denemede yönerge yalnızca sahne. `takvim[0]` listenin ilk satırı: Python saymaya 0'dan başlar.

In [ ]:
from uretim import ciz, sabirli_ciz, gonderi_yap, onay_sayfasi, goster, gunluge_ekle

pazartesi = takvim[0]
gorsel = ciz(pazartesi["sahne"], "ciktilar/deneme_gorsel.jpg")
gonderi = gonderi_yap(gorsel, pazartesi["baslik"], "gonderi", "ciktilar/deneme_gonderi.jpg")
goster(gonderi)

`ciktilar/` klasörüne bak: iki dosya var. `deneme_gorsel.jpg` modelin verdiği yazısız görsel; `deneme_gonderi.jpg`
Python'un ondan yaptığı gönderi.

**Başlığı neden modele yazdırmadık?** Görsel modelleri harfleri de resim gibi çizer: kelimeyi bozar, ş, ğ, ı, İ
gibi harfleri sık sık yanlış çizer ve fontu sen seçemezsin. Müşteri başlığı son dakika değiştirirse görseli baştan
üretmen gerekir. Tasarım programındaki şablonun mantığı burada da geçerli: görsel bir katman, yazı ayrı bir katman.
Yazıyı Python koyunca harfler hep doğru, font ve renk hep kılavuzdaki gibi.

Görsel Sayfa Kahve'ye benziyor mu? Kafenin kılavuzu iki renk istiyor (mürekkep mavisi ve kâğıt beyazı) ve iki
renkli risograf baskı tarzını. Bu görselde bunlar var mı?

## Adım 2 — Marka dili: stil cümlesi

Bir haftalık set yan yana durduğunda bir aile gibi görünmeli: Pazartesi'yi de Cuma'yı da gören kişi "Sayfa Kahve"
demeli. Buna **marka dili** diyoruz. Yapay zekâyı kendi hâline bırakırsan ortalama bir görsel verir; seti tutarlı
yapan senin kararların.

Arayüzlerde bunun için bir "stil referansı" görseli yüklersin. Kodda karşılığı **stil cümlesi** (`STIL`): kılavuzu
görsel modelinin anlayacağı dile çevirdiğin sabit bir cümle. Her sahnenin sonuna aynı cümle eklenir.

| Kılavuzda (`11-rag/veri/kilavuz.txt`) | Stil cümlesinde |
|---|---|
| İki renkli risograf baskı tarzı; mürekkep mavisi ve kâğıt beyazı | `two-color risograph print illustration, deep ink blue and warm paper white` |
| Üç boyutlu görsel ve parlak yüzey yok; hafif kayık baskı dokusu; alan boş ve sakin | `flat shapes, soft grain, plenty of empty space` |
| İnsan yüzü gösterilmez; yazıyı biz koyacağız | `no text, no letters, no faces` |

`STIL` büyük harfle yazılmış: "bu değer defter boyunca değişmeyecek" demek, bir gelenek.

Bu adımda **üretim günlüğünü** de başlatıyoruz: `gunluge_ekle(gunluk, gun, yonerge, dosya)`, `ciktilar/gunluk.csv`
dosyasına bir satır ekler; saati Python kendisi yazar. Neden gerektiğini hücreden sonra göreceğiz.

**Tahmin et, sohbete yaz:** Bu hücreyi bir kez daha çalıştırsan aynı Pazartesi görselini mi alırsın?

In [ ]:
STIL = "two-color risograph print illustration, deep ink blue and warm paper white"
STIL = STIL + ", flat shapes, soft grain, plenty of empty space"
STIL = STIL + ", no text, no letters, no faces"

yonerge = pazartesi["sahne"] + ", " + STIL
gorsel = ciz(yonerge, "ciktilar/Pazartesi_gorsel.jpg")
gonderi = gonderi_yap(gorsel, pazartesi["baslik"], "gonderi", "ciktilar/Pazartesi_gonderi.jpg")
gunluge_ekle("ciktilar/gunluk.csv", "Pazartesi", yonerge, gorsel)
goster(gonderi)

Adım 1'deki gönderiyle karşılaştır (yukarı kaydır ya da `ciktilar/` içinde ikisini yan yana aç). Renk ve teknik
kılavuza yaklaştı mı? Hangi kural tuttu, hangisi tutmadı? Tutmayanı cümleye ekleyebilir ya da daha açık
yazabilirsin. Ama karar şimdi: birazdan bütün hafta bu cümleyle üretilecek.

**Tahminine dönelim.** Hayır. Konu 7'de tohumu sabitleyip aynı görseli geri alabiliyorduk; bu model tohum almadığı
için aynı yönerge her seferinde başka bir görsel verir (Adım 5'te kendin göreceksin). Bunun iki sonucu var:

- Adım 1 ile buradaki farkın bir kısmı rastlantı; Konu 7'deki gibi tam bir kontrollü deney değil. Ama iki renk ve
  baskı tarzı stil cümlesinden geliyor: **tutarlılığı stil cümlesi sağlar.**
- Beğendiğin bir görseli yeniden üreterek geri alamazsın. Bu yüzden **dosyayı ve yönergeyi saklarız**; günlük bunun
  için. Bu hücreyi yeniden çalıştırırsan Pazartesi'nin üstüne yeni bir görsel yazılır: beğendiysen dokunma.

## Adım 3 — Bütün hafta

Pazartesi hazır; döngü kalan dört günü yapıyor. `takvim[1:]` bir **dilim**: 1 numaralı satırdan listenin sonuna
kadar. Python saymaya 0'dan başladığı için 0 numaralı Pazartesi dışarıda kalır, döngü Salı'dan Cuma'ya dört satırı
gezer. (Konu 1'deki `kelimeler[:8]` de bir dilimdi: baştan sekiz tane.) Döngü her turda bir satır alır: yönergeyi
kurar, görseli üretir, gönderiyi yapar, günlüğe yazar.

- `sabirli_ciz(yonerge, yol)`: `ciz`'in sabırlı hâli. Görsel gelmezse 5 saniye bekleyip yeniden dener (en fazla
  3 kez). İsteklerden biri takılırsa set yarım kalmasın diye.
- Dosya adını günden kuruyoruz (`f"ciktilar/{gun}_gonderi.jpg"`): her gün kendi dosyasına. Hepsine aynı adı
  verseydik her tur bir öncekinin üstüne yazardı.

**Tahmin et:** Bu döngü modele kaç istek gönderir?

In [ ]:
for satir in takvim[1:]:
    gun = satir["gun"]
    yonerge = satir["sahne"] + ", " + STIL
    gorsel = sabirli_ciz(yonerge, f"ciktilar/{gun}_gorsel.jpg")
    gonderi_yap(gorsel, satir["baslik"], "gonderi", f"ciktilar/{gun}_gonderi.jpg")
    gunluge_ekle("ciktilar/gunluk.csv", gun, yonerge, gorsel)
    print(gun, "hazır")

Dört istek, dört gönderi; Pazartesi'yle birlikte hafta tamam. Günlüğe bakalım:

In [ ]:
with open("ciktilar/gunluk.csv", encoding="utf-8-sig") as dosya:
    metin = dosya.read()

print(metin)

Her satır bir üretim: zaman, gün, tam yönerge, dosya. Müşteri "şunu değiştir" dediğinde hangi görselin hangi
yönergeyle üretildiğini ve yeni denemeye nereden başlayacağını buradan bilirsin. Bir hücreyi yeniden çalıştırırsan
görselin üstüne yenisi yazılır, günlük de bir satır uzar. Günlükten satır silinmez, yalnızca eklenir. Dosyayı
VS Code'da ya da bir tablo programında da açabilirsin.

## Adım 4 — Hikâyeler ve onay sayfası

Hikâyeler için yeni görsel üretmiyoruz: Python, Adım 2 ve 3'ün görsellerini 9:16'ya yerleştiriyor. Hikâyede başlık
üst ve alt kenara yaklaşmıyor; telefonda üst şeridin üstünde profil adı, alt şeridin üstünde yanıt kutusu durur.

Aynı döngüde iki liste de dolduruyoruz: gönderilerin ve hikâyelerin dosya adları. Onay sayfası bu listelerden kurulacak.

**Tahmin et:** Bu hücre modele kaç istek gönderir?

In [ ]:
gonderiler = []
hikayeler = []
for satir in takvim:
    gun = satir["gun"]
    gorsel = f"ciktilar/{gun}_gorsel.jpg"
    hikaye = gonderi_yap(gorsel, satir["baslik"], "hikaye", f"ciktilar/{gun}_hikaye.jpg")
    gonderiler.append(f"ciktilar/{gun}_gonderi.jpg")
    hikayeler.append(hikaye)

Sıfır istek: bu hücrede `ciz` yok. Şimdi müşteriye gidecek tek dosya. `onay_sayfasi(yollar, yol)` dosyaların küçük
hâllerini dizer, her birinin altına dosya adını yazar. `gonderiler + hikayeler` iki listeyi uç uca ekler: üst
sırada gönderiler, alt sırada hikâyeler.

In [ ]:
tum_dosyalar = gonderiler + hikayeler
onay_sayfasi(tum_dosyalar, "ciktilar/onay_sayfasi.jpg")
goster("ciktilar/onay_sayfasi.jpg")

Beş gün yan yana: bir aile gibi duruyorlar mı? Her küçük resmin altında dosyanın adı var; geri bildirim adıyla gelir,
hangi dosyayı değiştireceğin karışmaz. Büyük hâlini görmek için `ciktilar/onay_sayfasi.jpg` dosyasını VS Code'da aç.

## Adım 5 — Hangi gün olmamış?

Onay sayfası müşteriye gitmeden önce bir de sen bak, kılavuzun gözüyle. Sayfa Kahve'nin kılavuzu görseller için iki
kesin kural koyuyor, üçüncüsü bizim:

- renkler mürekkep mavisi ve kâğıt beyazı, en fazla küçük bir hardal sarısı vurgu; **kırmızı ve turuncu hiç yok**
- insan yüzü yok (eller olabilir)
- görselin içinde yazı ya da harf yok (başlığı Python koyuyor)

Stil cümlesi "deep ink blue and warm paper white" diyor; model yine de kendi alışkanlığına kayıp kılavuzun yasakladığı
bir rengi ekleyebilir. Kendi hâline bırakılan yapay zekâ ortalamaya kayar: seçmek ve "bu olmamış" demek tasarımcının işi.

**Sohbete yaz:** Senin onay sayfanda hangi gün kılavuza uymuyor, neden? Herkesin sayfası başka; bir arkadaşının
sorunlu günü seninkiyle aynı olmayabilir. Hepsi uyuyorsa en zayıf bulduğun günü seç.

Şimdi yalnızca o günü yenile. Bütün haftayı yeniden üretmek beş istek demek; üstelik iyi olan günler de değişir.
Aşağıdaki boş hücreye **kendin yaz**:

1. İlk satıra seçtiğin günü yaz: `gun = "Perşembe"` gibi. Takvimdeki yazımla, Türkçe harfleriyle.
2. O günün satırını takvimde bul: `for satir in takvim:`, içinde `if satir["gun"] == gun:`, onun içinde
   `secilen = satir`.
3. Yönergeyi Adım 3'teki gibi kur: `secilen["sahne"] + ", " + STIL`.
4. Görseli **yeni bir dosya adıyla** üret: `f"ciktilar/{gun}_gorsel_2.jpg"`. Eskisinin üstüne yazma: yenisi başka bir
   yerden bozuk çıkabilir ve eskisini bir daha üretemezsin.
5. Gönderiyi ve hikâyeyi de yeni adlarla yap (`_gonderi_2`, `_hikaye_2`), günlüğe yeni satırı ekle, gönderiyi göster.

Adım 3'teki döngünün içi yol gösterir; kopyalayıp değiştirmek serbest.

Yasağı yönergeye de yazabilirsin: `secilen["sahne"] + ", " + STIL + ", no orange, no red"`. Bu modelde ayrı bir
negatif yönerge alanı yok (Konu 7); yasak, yönergenin içinde bir istek olarak gider. Bazen işe yarar, bazen yaramaz:
dene ve gör.

**Tahmin et:** Yönergeyi hiç değiştirmezsen yeni görsel eskisinin aynısı mı olur?

Eskisiyle yenisini yan yana koy. Bu hücre senin hücrendeki `gun`'u kullanıyor:

In [ ]:
secenekler = [f"ciktilar/{gun}_gonderi.jpg", f"ciktilar/{gun}_gonderi_2.jpg"]
onay_sayfasi(secenekler, f"ciktilar/karsilastirma_{gun}.jpg")
goster(f"ciktilar/karsilastirma_{gun}.jpg")

**Düşün, sohbete yaz:** Yenisi kılavuza uydu mu? Uyduysa haftanın onay sayfasında da yenisi görünsün diye Adım 4'teki
hangi listeleri nasıl değiştirirsin? Uymadıysa ne yaparsın?

## Bonus — Kendi müşterin

Ödevde kendi müşterinle çalışacaksın: bir öğrenci kulübü, ailenin dükkânı, kendi markan ya da bir arkadaşının işletmesi.
Bugün küçük bir provası:

1. `veri/benim_takvimim.csv` dosyasını VS Code'da aç. İçinde örnek üç satır var (hayalî bir fotoğraf kulübü);
   onları kendi müşterinin üç günüyle değiştir. İlk satıra (`gun,baslik,sahne`) dokunma; sahneyi İngilizce, yazısız
   yaz ve tırnak içinde bırak.
2. Aşağıdaki hücrede `BENIM_STILIM` cümlesini kendi müşterinin görsel diline göre yeniden yaz.
3. İki hücreyi çalıştır: 3 görsel üretir; çıktılar `ciktilar/benim/` içine gider.

Kendi müşterinin renkleri için `gonderi_yap`'a iki bilgi daha verebilirsin: `zemin="#FFFFFF", renk="#222222"` gibi.

In [ ]:
with open("veri/benim_takvimim.csv", encoding="utf-8-sig") as dosya:
    okuyucu = csv.DictReader(dosya)
    benim_takvimim = list(okuyucu)

BENIM_STILIM = "bold paper cut-out collage, deep green and cream, simple shapes"
BENIM_STILIM = BENIM_STILIM + ", plenty of empty space, no text, no letters"

In [ ]:
benim_dosyalar = []
for satir in benim_takvimim:
    gun = satir["gun"]
    yonerge = satir["sahne"] + ", " + BENIM_STILIM
    gorsel = sabirli_ciz(yonerge, f"ciktilar/benim/{gun}_gorsel.jpg")
    gonderi = gonderi_yap(gorsel, satir["baslik"], "gonderi", f"ciktilar/benim/{gun}_gonderi.jpg")
    gunluge_ekle("ciktilar/benim/gunluk.csv", gun, yonerge, gorsel)
    benim_dosyalar.append(gonderi)

onay_sayfasi(benim_dosyalar, "ciktilar/benim/onay_sayfasi.jpg")
goster("ciktilar/benim/onay_sayfasi.jpg")

Üç gönderi bir aile gibi mi duruyor? Değilse stil cümlesini değiştirip yeniden çalıştır; günlük her denemeyi tutar.
Ödevde bunu beş günle yapacaksın (`odevler/odev8.md`).